In [55]:
from scipy.io import loadmat
import pandas as pd
import os
import re

subs=[num for num in os.listdir('../stimuli/logs/')]
partner_codes = {1:'Computer',2:'Stranger',3:'Friend'}
Button_codes = {2:'Right', 7:'Left', 0:'Miss'}
feedback_codes = {1:'Punish',2:'Neutral',3:'Reward'}

for sub in subs:
    print("running sub-%s"%(sub))
    eventfiles = ['../stimuli/logs/%s/%s'%(sub,file) for file in os.listdir('../stimuli/logs/%s'%(sub)) if file.endswith('raw.csv')]
    if int(sub)==10007:
        for file in eventfiles:
            try:
                x=pd.read_csv(file)
                scan_start=float(x['InitFixOnset'][0])
                x['Partner'] = x['Partner'].map(partner_codes).astype('str')
                x['Feedback'] = x['Feedback'].map(feedback_codes).astype('str')
                
                x['resp'] = x['resp'].map(Button_codes).astype('str')
                x.loc[x['resp']=='Miss','resp'] = 'Miss'
                
                x['feed_type'] = x[['Partner', 'Feedback']].agg('_'.join, axis=1)
                x.loc[x['resp']=='Miss','feed_type'] = 'MissFB'

                x['dec_type'] = x[['Partner', 'resp']].agg('_'.join, axis=1)
                x.loc[x['resp']=='Miss','dec_type'] = 'MissDC'
                display(x['resp'].unique())

                data=[]
                for index, row in x.iterrows(): #seperating out 2 kinds of events per file
                    feedback_info=[float(row['outcome_onset'])-scan_start,
                                 float(row['outcome_offset'])-float(row['outcome_onset']),
                                 row['feed_type'],
                                 row['rt']]

                    button_info=[float(row['decision_onset'])-scan_start,
                                   row['rt'],
                                   row['dec_type'],
                                   'n/a']
                    data.append(button_info)
                    data.append(feedback_info)
                df=pd.DataFrame(columns=[['onset','duration','trial_type','response_time']],data=data)
                
                outdir = '../bids/sub-%s/func'%(sub)
                if not os.path.exists(outdir):
                    os.makedirs(outdir)
                fullname = os.path.join(outdir,
                                        re.search('/%s/(.*)_raw'%(sub),file).group(1))+'_events.tsv'

                df.to_csv(fullname,sep='\t',index=False)
            except:
                print("Something went wrong for sub-%s file: %s"%(sub,file))

running sub-2002
running sub-10001
running sub-10010
running sub-10026
running sub-10008
running sub-2001
running sub-10007


array(['Left', 'Right'], dtype=object)

array(['Left', 'Right'], dtype=object)

array(['Right', 'Left'], dtype=object)

array(['Right', 'Left', 'Miss'], dtype=object)

array(['Right', 'Left'], dtype=object)

array(['Left', 'Right', 'Miss'], dtype=object)

running sub-10015
